In [ ]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import models, layers, regularizers
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [156]:
df = pd.read_csv('qoute_dataset.csv')

In [157]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   quote   3038 non-null   str  
 1   Author  3038 non-null   str  
dtypes: str(2)
memory usage: 554.0 KB


In [158]:
df.isnull().sum()

quote     0
Author    0
dtype: int64

In [159]:
df.duplicated().sum()

np.int64(0)

In [160]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [161]:
# we need to tokenize (smaller groups), and convert them into numbers, so we use vetorizer since it removes punctuation and lowercassing auto
# Limit max_tokens to 2500 to eliminate rare words that cause sparsity
x_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=5000,        #eliminates rare words that cause model sparsity, used as input dim
    output_sequence_length=30       #all columns same length be it 4 words or 10, the rest padded with 0
)
x_vectorizer.adapt(df['quote'].values)      #here df['quote'].lower() + translator=str.maketrans(" "," ",string.punctuation) #where (replaced, replaced with, deleted)       quotes=quotes.apply(lambda x:x.translate(translator))

# Regenerate your quote array with the capped vocabulary
quote = x_vectorizer(df['quote'].values).numpy()

In [162]:
# here we need to delete blanks seperately cuz they're padded with 0s or they'll be trained in the model
WINDOW_SIZE = 5  # This means the model looks at 5 words to predict the 6th

all_X = []      #create empty lists to be filled later wit the words in each row
all_y = []

for sequence in quote:      #word in a quote
    # Remove any trailing padding zeros so we only work with real words
    real_tokens = [token for token in sequence if token != 0]
    
    # Slice the quote into sliding window pairs
    for i in range(WINDOW_SIZE, len(real_tokens)):
        X_window = real_tokens[i - WINDOW_SIZE : i]
        y_target = real_tokens[i]
        
        all_X.append(X_window)
        all_y.append(y_target)

# Convert your fresh windows into numpy arrays
X_clean = np.array(all_X)
y_clean = np.array(all_y)

In [163]:
quote.shape

(3038, 30)

In [ ]:
# shuffle it so the val split doesn't take the same words only, to both ip and op
shuffle_map = np.random.permutation(len(X_clean))
X_shuffled = X_clean[shuffle_map]
y_shuffled = y_clean[shuffle_map]

In [165]:
print(len(x_vectorizer.get_vocabulary())) #we use this as input dim

5000


In [ ]:
rnn = models.Sequential([
    # Expanded embedding dim from 16 to 128
    layers.Embedding(input_dim=5000, output_dim=128, mask_zero=True),       #embeddinngs ceate a list of each row [2 3 9 100]
    
    layers.SimpleRNN(units=64, activation='tanh', return_sequences=True, recurrent_dropout=0.2),        #two rnn = return sequnces true on top ones
    layers.Dropout(0.3),        #regulate
    
    layers.SimpleRNN(units=64, activation='tanh', recurrent_dropout=0.2), 
    layers.Dropout(0.3),
    
    layers.Dense(units=5000, activation='softmax') 
])

In [167]:
rnn.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Stop training as soon as validation loss goes up for 2 epochs in a row, i don't wanna wait for all epochs if it's going badly
#relies on val loss to stop if 2 r going higher, and it returns the best accuracy instead of last
stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True) 

In [ ]:
rnn.fit(X_shuffled, y_shuffled, epochs=50, validation_split=0.2, callbacks=[stop])

Epoch 1/50
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - accuracy: 0.0423 - loss: 6.4309 - val_accuracy: 0.0589 - val_loss: 6.1960
Epoch 2/50
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.0613 - loss: 6.0477 - val_accuracy: 0.0670 - val_loss: 6.0950
Epoch 3/50
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.0764 - loss: 5.8399 - val_accuracy: 0.0849 - val_loss: 6.0036
Epoch 4/50
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.0926 - loss: 5.6342 - val_accuracy: 0.0958 - val_loss: 5.9579
Epoch 5/50
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.1067 - loss: 5.4379 - val_accuracy: 0.0999 - val_loss: 5.9255
Epoch 6/50
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.1185 - loss: 5.2769 - val_accuracy: 0.1026 - val_loss: 5.9350
Epoch 7/50
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.1259 - loss: 5.1353 - val_accuracy: 0.1056 - val_loss: 5.9557


In [ ]:
# the more the dataset complexity the higher the chance rnn forgets the previous steps(vanishing gradient descent), so we use long-short term memory to keep track of it (LSTM)
#all the same just swap simplernn with lstm
lstm = models.Sequential([
    # Expand embedding dimensions so 5000 words can map out clearly
    layers.Embedding(input_dim=5000, output_dim=128, mask_zero=True), 
    
    # Give the LSTM layer enough units (128) to actually utilize its gates
    layers.LSTM(units=128, activation='tanh', return_sequences=True, recurrent_dropout=0.2),
    layers.Dropout(0.3),
    
    layers.LSTM(units=128, activation='tanh', recurrent_dropout=0.2),
    layers.Dropout(0.3),
    
    layers.Dense(units=5000, activation='softmax')
])

In [ ]:
lstm.compile(optimizer= Adam(learning_rate=0.0005), loss='sparse_categorical_crossentropy', metrics=['accuracy']) #higher lr cuz it can handle it

In [ ]:
lstm.fit(X_shuffled, y_shuffled, epochs=100, validation_split=0.2, callbacks=[stop])

Epoch 1/100
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 30s 23ms/step - accuracy: 0.0401 - loss: 6.3878 - val_accuracy: 0.0445 - val_loss: 6.1777
Epoch 2/100
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 22s 21ms/step - accuracy: 0.0492 - loss: 6.0593 - val_accuracy: 0.0592 - val_loss: 6.1692
Epoch 3/100
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step - accuracy: 0.0586 - loss: 5.9434 - val_accuracy: 0.0708 - val_loss: 6.1131
Epoch 4/100
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step - accuracy: 0.0687 - loss: 5.8251 - val_accuracy: 0.0674 - val_loss: 6.0799
Epoch 5/100
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 23s 21ms/step - accuracy: 0.0737 - loss: 5.7311 - val_accuracy: 0.0716 - val_loss: 6.0738
Epoch 6/100
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step - accuracy: 0.0776 - loss: 5.6512 - val_accuracy: 0.0751 - val_loss: 6.0625
Epoch 7/100
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step - accuracy: 0.0835 - loss: 5.5670 - val_accuracy: 0.0795 - val_loss: 6.0895
Epoch 8/100
1067/1067 ━━━━━━━━━━━━━━━━━━━━ 23s 21ms/step - accuracy: 

In [173]:
# there's also GRU (gated recurrent unit), same code but replace lstm with gru
#  it uses two gates instead of three unlike lstm to keep track of the prev steps
#slightly better and faster epoch

In [ ]:
# now create nlp model to predisct te next word
import numpy as np

def predict(text, next_words_count, model, vectorizer = x_vectorizer):
    generated= text     # takes my sentnnce and makes it output
    vocabulary = vectorizer.get_vocabulary()
    vocab_len = len(vocabulary)  # Find the exact size of your vocabulary list
    
    # to predict more than one word create a loop
    for _ in range(next_words_count):
        tokenized_input = vectorizer([generated]).numpy()       #make them arrays too
        input_sequence = tokenized_input[:, -5:]        #cceks 5 prev words to deduct, excludes the last
        
        # prediction probabilities
        predictions = model.predict(input_sequence, verbose=0)[0]       #verbrose hides that loading line only, [0] is needed bcz we dont want nested loop array
        
        # match the EXACT length of the vocabulary (it chooses from those 10k words)
        predictions = predictions[:vocab_len]
        
        # Explicitly zero out the safety tokens if they exist in our range (we dont want blank or unkown pred)
        if "" in vocabulary:
            predictions[vocabulary.index("")] = 0
        if "[UNK]" in vocabulary:
            predictions[vocabulary.index("[UNK]")] = 0
            
        # Normalize after zeroing out and slicing
        if np.sum(predictions) > 0:     #creates probability of non zeroes only, the highest prob is chosen
            predictions = predictions / np.sum(predictions)
            
        # Apply temperature, this sticks out higher prob over others, while also giving chance to the less ones then normalize them back to prob at the end
        temperature = 0.7
        predictions = np.log(predictions + 1e-7) / temperature
        exp_preds = np.exp(predictions)
        scaled_probabilities = exp_preds / np.sum(exp_preds)
        
        # Sample randomly the words in range, higher gets better chnace but lower stil has a chnce to be picked too
        predicted_index = np.random.choice(len(scaled_probabilities), p=scaled_probabilities)
        predicted_word = vocabulary[predicted_index]
        
        generated += " " + predicted_word       #takes te generated word and with space from the prev word
        
    return generated

In [ ]:
# Pass it a prompt and ask for 6 words back
print(predict("life is a", 6, rnn, x_vectorizer))

life is a longer natural our determines thisim forgetting


In [ ]:
#now with lstm
print(predict("life is a", 6, lstm, x_vectorizer))

life is a soul talks multitude kindness beautiful enjoying
